# Brazilian E-commerce Sales and Fulfillment Analysis, 2017-2018: Growth, Seasonality, Delivery vs. Estimates

### Whats measurable with this dataset:

1. Initial Exploration & Data Characteristics

    * **Q1.1:** What are the data types of all columns in the 8 tables?
    * **Q1.2:** What is the overall time range during which orders were placed, and which periods are excluded?
    * **Q1.3:** What is the total count of distinct cities and states where ordering customers are located?
    * **Q1.4 (guardrail):** What is the order status mix and are there date inconsistencies?

2. In-Depth Exploration & Order Trends

    * **Q2.1:** Is there a growing trend in the number of orders placed over time?
    * **Q2.2:** How do order volumes vary across months of the year, split by year (seasonality)?
    * **Q2.3:** How do order volumes vary across hours and time windows of the day?
    * **Q2.4:** How do order volumes vary across days of the week?

3. Evolution of E-Commerce in Brazil

    * **Q3.1:** What is the Month-on-Month (MoM) order count across customer states?
    * **Q3.2:** How are orders and unique customers distributed across states?

4. Impact on Economy: Price & Freight Analysis

    * **Q4.1:** What is the percentage change in order value (GMV) and freight between 2017 and 2018?
    * **Q4.2:** What are the total and average order prices per customer state?
    * **Q4.3:** What are the total and average freight costs per order per customer state, and is freight lower where order value is higher?

5. Delivery Performance & Freight Analysis

    * **Q5.1:** What is the delivery time distribution, the delivery gap, the on-time rate and its effect on review scores?
    * **Q5.2:** Which are the 5 states with the highest and lowest average freight per order?
    * **Q5.3:** Which are the 5 states with the slowest and fastest average delivery times?
    * **Q5.4:** Which 5 states deliver earliest relative to the estimate, and which have the lowest on-time rate?
    * **Q5.5 (supplementary):** Is the freight/delivery advantage explained by seller location or by volume?

6. Payment Analysis

    * **Q6.1:** What is the Month-on-Month order count for each payment type?
    * **Q6.2:** What is the total number of orders per year for each payment type?
    * **Q6.3:** How are orders distributed by number of payment installments?

7. Key Metrics Summary

8. Revenue by Category

    * Which categories drive the most GMV, and what share do the top 10 hold?
    * How does category mix shift between Jan–Aug 2017 and Jan–Aug 2018?
    * Average price, freight as a % of price, and review score by category.
    * Do heavy or bulky categories have worse delivery times and freight burden?

### Not measurable with this dataset (no data):
Net profit margin, CAC, cart abandonment, revenue per visitor, inventory turnover, true return/refund rate, true LTV. `freight_value` is the shipping charge on an item, **not** a cost to Target, so no margin conclusions are drawn from it.

## **Q1.1:** What are the data types of all columns in the 8 tables?

```sql
select
	table_name, 
	column_name,
	data_type
from
	information_schema.columns
where
	table_schema = "brazil_ecommerce"
	and table_name in (
		"customers_view",
		"geolocation_view",
		"order_items_view",
		"payments_view",
		"order_reviews_view",
		"orders_view",
		"products_view",
		"sellers_view"
	)
order by
	table_name,
	ordinal_position;
```

---

Insights:
  - 8 tables and 49 columns profiled.
  - 3 date/time columns outside `orders` are stored as text (VARCHAR) and must be cast before any date arithmetic: shipping_limit_date, review_creation_date, review_answer_timestamp
  - `orders` timestamps were cast to TIMESTAMP at load time (TRY_CAST turns unparseable values into NULL).

## **Q1.2:** What is the overall time range during which orders were placed, and which periods are excluded?

```sql
select
	min(order_purchase_timestamp) as `min_date`,
	max(order_purchase_timestamp) as `max_date`,
	count(*) as `total_orders`
from
	orders_view
where
	order_purchase_timestamp is not null;
```

---

```sql
select
	date_format(order_purchase_timestamp, "%Y-%m") as `year_month`,
	count(*) as `order_count`
from
	orders_view
where
	order_purchase_timestamp between 
	"2017-01-01 00:00:00" and "2018-09-01 23:59:59"
group by
	`year_month`
order by
	`year_month`;
```

---

```sql
alter view orders_view as 
select
    *
from
    orders_view
where
    order_purchase_timestamp between 
	"2017-01-01 00:00:00" and "2018-09-01 23:59:59"
```

---

Insights:
  - Orders span 2016-09-04 to 2018-10-17 (99,441 orders).
  - Analysis window is Jan 2017 to Aug 2018. 349 orders (0.35%) fall outside it and are excluded.
  - Excluded months (partial or incomplete coverage): 2016-09, 2016-10, 2016-12, 2018-09, 2018-10
  - Only two calendar years remain, so a yearly trend cannot be established; trend is assessed on monthly data and Jan-Aug vs Jan-Aug.

## **Q1.3:** What is the total count of distinct cities and states where ordering customers are located?

```sql
select
	count(distinct concat(
		customer_state, "_", 	
		lower(trim(customer_city))
	)) as city_count,
	count(distinct customer_state) as state_count
from
	customers_view as c inner join orders_view as o
	on c.customer_id = o.customer_id;
```

---

Insights:
  - Orders come from 27 states (27 expected incl. the Federal District) and 4,310 distinct cities.
  - Cities are counted as unique (state, lower-cased trimmed name) pairs, so identical names in different states are kept apart.

## **Q1.4 (guardrail):** What is the order status mix and are there date inconsistencies?

```sql
select
	order_status,
	count(*) as order_count,
	round(
		100.0 * count(*) / sum(count(*)) over(), 
	2) as pct_of_orders
from
	orders_view
group by
	order_status
order by
	order_count desc;
```

---

```sql
with cte1 as (
	select
		(
			order_status = "delivered" and 
			order_delivered_customer_date is null
		) as delivered_without_delivery_date,
		(
			order_delivered_customer_date is not null and
			order_purchase_timestamp is not null and
			order_delivered_customer_date < order_purchase_timestamp
		) as delivery_before_purchase,
		(
			order_status != "delivered" and
			order_delivered_customer_date is not null
		) as non_delivered_with_delivery_date
	from
		orders_view
	)
select
	sum(if(delivered_without_delivery_date = 1, 1, 0)) as delivered_without_delivery_date_count,
	sum(if(delivery_before_purchase = 1, 1, 0)) as delivered_without_delivery_date_count,
	sum(if(non_delivered_with_delivery_date = 1, 1, 0)) as delivered_without_delivery_date_count
from
	cte1;
```

---

Insights:
  - Delivered orders are 97.09% of orders placed in the window.
  - Canceled + unavailable orders are 1.20%; they are excluded from value/freight metrics and all delivery metrics.
  - Other non-delivered statuses (shipped, processing, invoiced, approved, created) are excluded from delivery metrics because they have no final delivery date.
  - Date checks: 8 delivered orders lack a delivery date, 0 show delivery before purchase (removed from delivery metrics).

## **Q2.1:** Is there a growing trend in the number of orders placed over time?

```sql
with cte1 as (
	select
		date_format(order_purchase_timestamp, "%Y-%m") as year_monthh,
		count(order_id) as total_orders
	from
		orders_view
	group by
		year_monthh
)
select
	year_monthh,
	total_orders,
	round(
		100.0 * (total_orders - lag(total_orders) over(order by year_monthh)) / nullif(
			lag(total_orders) over(order by year_monthh),
			0
		),
		2
	) as mom_pct_change
from
	cte1
order by
	year_monthh;
```

---

```sql
select
	extract(year from order_purchase_timestamp) as yearr,
	count(order_id) as jan_aug_orders
from
	orders_view
where
	extract(month from order_purchase_timestamp) between 1 and 8
group by
	yearr
order by
	yearr;
```

---

Insights:
  - Like-for-like (Jan-Aug) orders: 22,968 in 2017 vs 53,991 in 2018 (+135.1%).
  - Peak month: 2017-11 with 7,544 orders.
  - Average MoM change: 15.8% (median 3.5%).
  - A calendar-year comparison (12 months vs 8 months) would understate growth, so it is not used.


**Limitation:** with only two years of usable data, a yearly trend cannot be established. Evidence of growth rests on the 20-month series and the Jan-Aug comparison.

## **Q2.2:** How do order volumes vary across months of the year, split by year (seasonality)?

```sql
select
	extract(year from order_purchase_timestamp) as yearr,
	extract(month from order_purchase_timestamp) as monthh,
	count(order_id) as total_orders
from
	orders_view
group by 
	yearr, 
	monthh
order by
	yearr,
	monthh;
```

---

Insights:
  - Peak month by year: 2017: Nov, 2018: Jan.
  - Average orders per month in Jan-Aug: 2017: 2,871, 2018: 6,749.
  - Sep-Dec exist for 2017 only, so seasonality for those months cannot be compared across years.
  - Nov 2017 vs Oct 2017: +62.9% (a sharp jump, consistent with Black Friday).

## **Q2.3:** How do order volumes vary across hours and time windows of the day?

```sql
with cte1 as (
	select
		extract(hour from order_purchase_timestamp) as purchase_hour,
		count(*) as order_count
	from
		orders_view
	group by
		purchase_hour
)
select
	purchase_hour,
	case
		when purchase_hour between 0 and 6 then "Dawn"
		when purchase_hour between 7 and 12 then "Morning"
		when purchase_hour between 13 and 18 then "Afternoon"
		else "Night"
	end as time_of_day,
	order_count
from
	cte1
order by
	purchase_hour;
```

---

```sql
with cte1 as (
	select
		case
			when extract(hour from order_purchase_timestamp) between 0 and 6 then "Dawn"
			when extract(hour from order_purchase_timestamp) between 7 and 12 then "Morning"
			when extract(hour from order_purchase_timestamp) between 13 and 18 then "Afternoon"
			else "Night"
		end as time_of_day
	from
		orders_view
)
select
	time_of_day,
	count(*) as order_count,
	round(
		100.0 * count(*) / sum(count(*)) over(),
		2
	) as pct_of_orders
from
	cte1
group by
	time_of_day
order by
	order_count desc;
```

---

Insights:
  - Busiest time window: Afternoon (38.4% of orders); quietest: Dawn (5.3%).
  - Peak hour: 16:00 with 6,655 orders.
  - Windows: Dawn 00-06, Morning 07-12, Afternoon 13-18, Night 19-23.

## **Q2.4:** How do order volumes vary across days of the week?

```sql
select
	day(order_purchase_timestamp) as day_number,
	dayname(order_purchase_timestamp) as day_of_week,
	count(*) as order_count,
	round(
		100.0 * count(*) / sum(count(*)) over(),
		2
	) as pct_of_orders
from
	orders_view
group by
	day_number,
	day_of_week
order by
	day_number;
```

---

Insights:
  - Busiest day: Monday (16.3% of orders); quietest: Saturday (10.9%).
  - Weekend (Sat+Sun) share: 23.0% of orders vs 28.6% if volume were evenly spread.

## **Q3.1:** What is the Month-on-Month (MoM) order count across customer states?

```sql
select
	c.customer_state,
	c.customer_state_name,
	date_format(o.order_purchase_timestamp, "%Y-%m") as year_monthh,
	count(o.order_id) as order_count
from
	customers_view as c inner join orders_view as o
	on c.customer_id = o.customer_id
group by
	c.customer_state,
	c.customer_state_name,
	year_monthh
order by
	year_monthh,
	c.customer_state;
```

---

Insights:
  - Top 5 states (São Paulo, Rio de Janeiro, Minas Gerais, Rio Grande do Sul, Paraná) account for 77.2% of orders in the window.
  - 3 of 27 states recorded at least one month with zero orders out of 20, so small-state monthly series are noisy.

## **Q3.2:** How are orders and unique customers distributed across states?

```sql
select
	c.customer_state,
	c.customer_state_name,
	count(o.order_id) as total_orders,
	count(distinct c.customer_unique_id) as total_customers,
	round(
		100.0 * count(o.order_id) / sum(count(o.order_id)) over(),
		2
	) as pct_of_orders
from
	customers_view as c inner join orders_view as o
	on c.customer_id = o.customer_id
group by
	c.customer_state,
	c.customer_state_name
order by
	total_orders desc;
```

---

Insights:
  - Top 3 states: São Paulo (42.0%), Rio de Janeiro (12.9%), Minas Gerais (11.7%) = 66.6% of orders.
  - 15 of 27 states each contribute under 1% of orders.

## **Q4.1:** What is the percentage change in order value (GMV) and freight between 2017 and 2018?

```sql
WITH periods AS (
    SELECT 'Jan-Aug' AS period, order_year, order_price, order_freight FROM order_values_view WHERE order_month BETWEEN 1 AND 8
    UNION ALL
    SELECT 'Aug' AS period, order_year, order_price, order_freight FROM order_values_view WHERE order_month = 8
)
SELECT
    period,
    ROUND(SUM(IF(order_year = 2017, order_price, 0)), 2) AS gmv_2017,
    ROUND(SUM(IF(order_year = 2018, order_price, 0)), 2) AS gmv_2018,
    ROUND(100.0 * (SUM(IF(order_year = 2018, order_price, 0)) - SUM(IF(order_year = 2017, order_price, 0)))
            / NULLIF(SUM(IF(order_year = 2017, order_price, 0)), 0), 2) AS gmv_pct_change,
    ROUND(SUM(IF(order_year = 2017, order_freight, 0)), 2) AS freight_2017,
    ROUND(SUM(IF(order_year = 2018, order_freight, 0)), 2) AS freight_2018,
    ROUND(100.0 * (SUM(IF(order_year = 2018, order_freight, 0)) - SUM(IF(order_year = 2017, order_freight, 0)))
            / NULLIF(SUM(IF(order_year = 2017, order_freight, 0)), 0), 2) AS freight_pct_change
FROM periods
GROUP BY period
ORDER BY period ASC;
```

---

Insights:
  - Jan-Aug GMV rose +138.3% (BRL 3,080,851 to BRL 7,341,037); freight charged rose +153.4%.
  - August 2017 vs August 2018: GMV +49.4%, freight +58.1%.
  - Values are nominal BRL (not inflation-adjusted) and cover valid orders only (canceled/unavailable excluded).

## **Q4.2:** What are the total and average order prices per customer state?

```sql
WITH ranked AS (
    SELECT
        customer_state,
        customer_state_name,
        order_price,
        ROW_NUMBER() OVER (
            PARTITION BY customer_state, customer_state_name
            ORDER BY order_price
        ) AS rn,
        COUNT(*) OVER (
            PARTITION BY customer_state, customer_state_name
        ) AS cnt
    FROM order_values_view
    WHERE order_price IS NOT NULL
)
SELECT
    customer_state,
    customer_state_name,
    COUNT(*) AS order_count,
    ROUND(SUM(order_price), 2) AS total_order_price,
    ROUND(AVG(order_price), 2) AS avg_order_price,
    ROUND(AVG(CASE
        WHEN rn IN (FLOOR((cnt + 1) / 2), CEIL((cnt + 1) / 2))
        THEN order_price
    END), 2) AS median_order_price
FROM ranked
GROUP BY customer_state, customer_state_name
ORDER BY total_order_price DESC;
```

---

Insights:
  - São Paulo has the highest total order value (38.3% of national GMV) with an average order value of BRL 125.56 (rank 27 of 27).
  - Highest average order value (states with >= 100 orders): Paraíba (BRL 216.65); lowest: São Paulo (BRL 125.56).
  - Average order value ranges from BRL 125.56 to BRL 216.65 across ranked states (73% spread), while total value spans a far wider range because it follows order volume.

## **Q4.3:** What are the total and average freight costs per order per customer state, and is freight lower where order value is higher?

```sql
WITH ranked AS (
    SELECT
        customer_state,
        customer_state_name,
        order_freight,
        order_price,
        ROW_NUMBER() OVER (
            PARTITION BY customer_state, customer_state_name
            ORDER BY order_freight
        ) AS rn,
        COUNT(*) OVER (
            PARTITION BY customer_state, customer_state_name
        ) AS cnt
    FROM order_values_view
    WHERE order_freight IS NOT NULL
)
SELECT
    customer_state,
    customer_state_name,
    COUNT(*) AS order_count,
    ROUND(SUM(order_freight), 2) AS total_freight_value,
    ROUND(AVG(order_freight), 2) AS avg_freight_per_order,
    ROUND(AVG(CASE
        WHEN rn IN (FLOOR((cnt + 1) / 2), CEIL((cnt + 1) / 2))
        THEN order_freight
    END), 2) AS median_freight_per_order,
    ROUND(100.0 * SUM(order_freight) / SUM(order_price), 2) AS freight_pct_of_price
FROM ranked
GROUP BY customer_state, customer_state_name
ORDER BY avg_freight_per_order ASC;
```

---

Insights:
  - Lowest average freight per order: São Paulo (BRL 17.37, 13.8% of price). Highest: Roraima (BRL 49.23, 27.8% of price, 43 orders).
  - Top 5 states by total order value and their freight rank: São Paulo (value #1, freight #1 lowest); Rio de Janeiro (value #2, freight #5 lowest); Minas Gerais (value #3, freight #2 lowest); Rio Grande do Sul (value #4, freight #8 lowest); Paraná (value #5, freight #3 lowest).
  - Rank correlation between average order value and average freight across states: +0.88 (near 0 = unrelated).
  - Freight is the shipping charge on the order, not Target's cost; this data cannot show margin. Use freight as a share of price as the affordability measure.

## **Q5.1:** What is the delivery time distribution, the delivery gap, the on-time rate and its effect on review scores?

```sql
SELECT
    order_id,
    time_to_deliver_days,
    promised_days,
    diff_estimated_delivery
FROM delivery_metrics_view;
```

---

```sql
-- MEDIAN(time_to_deliver_days)
-- MEDIAN(diff_estimated_delivery)
SELECT
    COUNT(*) AS delivered_orders,
    ROUND(AVG(time_to_deliver_days), 2) AS mean_delivery_days,
    ROUND(AVG(promised_days), 2) AS mean_promised_days,
    ROUND(AVG(diff_estimated_delivery), 2) AS mean_gap_days,
    ROUND(100.0 * AVG(CASE WHEN diff_estimated_delivery >= 0 THEN 1 ELSE 0 END), 2) AS on_time_or_early_pct,
    ROUND(100.0 * AVG(CASE WHEN diff_estimated_delivery < 0 THEN 1 ELSE 0 END), 2) AS late_pct,
    ROUND(100.0 * AVG(CASE WHEN diff_estimated_delivery < -7 THEN 1 ELSE 0 END), 2) AS late_over_7_days_pct
FROM delivery_metrics_view;
```

---

```sql
-- MEDIAN(r.review_score)
WITH review_scores AS (
    SELECT order_id, AVG(review_score) AS review_score
    FROM reviews_view
    GROUP BY order_id
)
SELECT
    CASE WHEN d.diff_estimated_delivery >= 0 THEN 'On Time or Early' ELSE 'Late' END AS delivery_status,
    COUNT(*) AS orders_with_review,
    ROUND(AVG(r.review_score), 2) AS avg_review_score,
    ROUND(100.0 * AVG(CASE WHEN r.review_score <= 2 THEN 1 ELSE 0 END), 2) AS low_score_pct
FROM delivery_metrics_view d
JOIN review_scores r ON d.order_id = r.order_id
GROUP BY delivery_status
ORDER BY delivery_status DESC;
```

---

Insights:
  - 96,203 delivered orders: mean delivery time 12.54 days (median 10.21) against a mean promise of 24.28 days.
  - On average orders arrive 11.81 days early (median 12.0), but 6.79% arrive late and 2.97% arrive more than 7 days late. The mean gap hides these late orders.
  - Average review score is 4.29 for on-time/early vs 2.27 for late orders; low scores (1-2) are 9.21% vs 62.35%.

## **Q5.2:** Which are the 5 states with the highest and lowest average freight per order?

```sql
WITH state_freight AS (
    SELECT
        customer_state,
        COUNT(*) AS order_count,
        ROUND(AVG(order_freight), 2) AS avg_freight_per_order,
        ROUND(100.0 * SUM(order_freight) / SUM(order_price), 2) AS freight_pct_of_price
    FROM order_values_view
    GROUP BY customer_state
    HAVING COUNT(*) >= 1000
),
ranked AS (
    SELECT *,
        ROW_NUMBER() OVER (ORDER BY avg_freight_per_order DESC) AS rank_high,
        ROW_NUMBER() OVER (ORDER BY avg_freight_per_order ASC) AS rank_low
    FROM state_freight
)
SELECT customer_state, order_count, avg_freight_per_order, freight_pct_of_price, 'Highest 5' AS category, rank_high AS rank_in_group
FROM ranked WHERE rank_high <= 5
UNION ALL
SELECT customer_state, order_count, avg_freight_per_order, freight_pct_of_price, 'Lowest 5' AS category, rank_low AS rank_in_group
FROM ranked WHERE rank_low <= 5
ORDER BY category DESC, rank_in_group ASC;
```

---

Insights:
  - Highest freight per order: Paraíba (BRL 48.35), Rondônia (BRL 46.31), Piauí (BRL 43.09), Maranhão (BRL 42.49), Tocantins (BRL 42.07).
  - Lowest freight per order: São Paulo (BRL 17.37), Minas Gerais (BRL 23.46), Paraná (BRL 23.55), Distrito Federal (BRL 23.79), Rio de Janeiro (BRL 23.95).
  - National average is BRL 22.82; the most expensive ranked state costs 2.8x the cheapest.
  - States below 100 orders were not ranked: AC, AP, RR.

## **Q5.3:** Which are the 5 states with the slowest and fastest average delivery times?

```sql
-- MEDIAN(time_to_deliver_days)
WITH state_delivery AS (
    SELECT
        customer_state,
        COUNT(*) AS order_count,
        ROUND(AVG(time_to_deliver_days), 2) AS avg_delivery_days
    FROM delivery_metrics_view
    GROUP BY customer_state
    HAVING COUNT(*) >= 1000
),
ranked AS (
    SELECT *,
        ROW_NUMBER() OVER (ORDER BY avg_delivery_days DESC) AS rank_slow,
        ROW_NUMBER() OVER (ORDER BY avg_delivery_days ASC) AS rank_fast
    FROM state_delivery
)
SELECT customer_state, order_count, avg_delivery_days, 'Slowest 5' AS category, rank_slow AS rank_in_group
FROM ranked WHERE rank_slow <= 5
UNION ALL
SELECT customer_state, order_count, avg_delivery_days, 'Fastest 5' AS category, rank_fast AS rank_in_group
FROM ranked WHERE rank_fast <= 5
ORDER BY category DESC, rank_in_group ASC;
```

---

```sql
SELECT
    customer_state,
    COUNT(*) AS order_count,
    ROUND(AVG(time_to_deliver_days), 2) AS avg_delivery_days
FROM delivery_metrics_view
GROUP BY customer_state
HAVING COUNT(*) >= 1000
ORDER BY avg_delivery_days ASC;
```

---

Insights:
  - Slowest: Amazonas (26.43 d, median 25.88), Alagoas (24.52 d, median 22.33), Pará (23.76 d, median 21.07), Maranhão (21.54 d, median 19.18), Sergipe (21.44 d, median 17.93).
  - Fastest: São Paulo (8.74 d, median 7.2), Paraná (11.97 d, median 10.42), Minas Gerais (11.98 d, median 10.31), Distrito Federal (12.94 d, median 11.36), Santa Catarina (14.95 d, median 13.01).
  - National average is 12.54 days; the slowest ranked state is 3.0x the fastest.
  - Green bars = fastest 5, red bars = slowest 5, grey = other states.

## **Q5.4:** Which 5 states deliver earliest relative to the estimate, and which have the lowest on-time rate?

```sql
-- MEDIAN(diff_estimated_delivery)
SELECT
    customer_state,
    COUNT(*) AS order_count,
    ROUND(AVG(diff_estimated_delivery), 2) AS avg_days_ahead_of_estimate,
    ROUND(100.0 * AVG(CASE WHEN diff_estimated_delivery >= 0 THEN 1 ELSE 0 END), 2) AS on_time_or_early_pct
FROM delivery_metrics_view
GROUP BY customer_state
HAVING COUNT(*) >= 1000
ORDER BY on_time_or_early_pct ASC;
```

---

```sql
-- MEDIAN(diff_estimated_delivery)
SELECT
    customer_state,
    COUNT(*) AS order_count,
    ROUND(AVG(diff_estimated_delivery), 2) AS avg_days_ahead_of_estimate,
    ROUND(100.0 * AVG(CASE WHEN diff_estimated_delivery >= 0 THEN 1 ELSE 0 END), 2) AS on_time_or_early_pct
FROM delivery_metrics_view
GROUP BY customer_state
HAVING COUNT(*) >= 1000
ORDER BY avg_days_ahead_of_estimate DESC
LIMIT 5;
```

---

Insights:
  - Earliest vs estimate: Rondônia (20.1 d ahead), Amazonas (19.57 d ahead), Mato Grosso (14.32 d ahead), Pará (13.96 d ahead), Rio Grande do Sul (13.84 d ahead).
  - Lowest on-time rate: Alagoas (78.54%), Maranhão (82.47%), Sergipe (84.64%), Piauí (86.11%), Ceará (86.17%) vs national 93.2%.
  - Large positive gaps show conservative estimates (expectations are exceeded). The on-time rate shows how often the promise was still broken, so both are read together.

## **Q5.5 (supplementary):** Is the freight/delivery advantage explained by seller location or by volume?

```sql
WITH order_origin AS (
    SELECT
        oi.order_id,
        MIN(s.seller_state = c.customer_state) AS same_state,
        SUM(oi.price) AS order_price,
        SUM(oi.freight_value) AS order_freight
    FROM order_items_view oi
    JOIN sellers_view s ON oi.seller_id = s.seller_id
    JOIN orders_view o ON oi.order_id = o.order_id
    JOIN customers_view c ON o.customer_id = c.customer_id
    GROUP BY oi.order_id
)
SELECT
    CASE WHEN oo.same_state = 1 THEN 'Seller in Same State' ELSE 'Seller in Another State' END AS seller_location,
    COUNT(*) AS order_count,
    ROUND(AVG(oo.order_freight), 2) AS avg_freight_per_order,
    ROUND(100.0 * SUM(oo.order_freight) / SUM(oo.order_price), 2) AS freight_pct_of_price,
    ROUND(AVG(d.time_to_deliver_days), 2) AS avg_delivery_days
FROM order_origin oo
LEFT JOIN delivery_metrics_view d ON oo.order_id = d.order_id
GROUP BY seller_location
ORDER BY seller_location ASC;
```

---

```sql
SELECT
    s.seller_state,
    COUNT(*) AS seller_count,
    ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 2) AS pct_of_sellers
FROM sellers_view s
GROUP BY s.seller_state
ORDER BY seller_count DESC;
```

---

Insights:
  - Same-state orders: BRL 15.36 freight, 7.92 days. Cross-state orders: BRL 26.98 freight, 15.12 days (35.8% of orders are same-state).
  - São Paulo holds 59.7% of all sellers.
  - If same-state orders are much cheaper and faster, the freight/delivery advantage of seller-dense states is mostly proximity, not volume economies of scale. Product weight and mix are not controlled for here.

## **Q6.1:** What is the Month-on-Month order count for each payment type?

```sql
SELECT
    DATE_FORMAT(o.order_purchase_timestamp, '%Y-%m') AS year_monthh,
    p.payment_type,
    COUNT(DISTINCT o.order_id) AS total_orders
FROM orders_view o
JOIN payments p ON o.order_id = p.order_id
GROUP BY year_monthh, p.payment_type
ORDER BY total_orders DESC, year_monthh ASC;
```

---

Insights:
  - Order counts by payment type over the window: Credit Card 76,248 (76.9%), UPI 19,721 (19.9%), Voucher 3,836 (3.9%), Debit Card 1,526 (1.5%), Not Defined 2 (0.0%).
  - Shares sum to more than 100% because 99,092 orders can use more than one payment type (e.g., voucher + credit card).
  - The file labels this category `UPI`; the problem statement calls it Boleto. Confirm the mapping before presenting.

## **Q6.2:** What is the total number of orders per year for each payment type?

```sql
WITH order_payments AS (
    SELECT
        EXTRACT(YEAR FROM o.order_purchase_timestamp) AS year,
        o.order_id,
        p.payment_type
    FROM orders_view o
    JOIN payments_view p ON o.order_id = p.order_id
),
year_totals AS (
    SELECT year, COUNT(DISTINCT order_id) AS orders_in_year
    FROM order_payments
    GROUP BY year
)
SELECT
    op.year,
    op.payment_type,
    COUNT(DISTINCT op.order_id) AS total_orders,
    ROUND(100.0 * COUNT(DISTINCT op.order_id) / MAX(yt.orders_in_year), 2) AS pct_of_year_orders
FROM order_payments op
JOIN year_totals yt ON op.year = yt.year
GROUP BY op.year, op.payment_type
ORDER BY op.year ASC, total_orders DESC;
```

---

Insights:
  - Share of orders using each payment type (2017 / 2018): UPI 21.1% / 18.9%; Credit Card 76.3% / 77.5%; Debit Card 0.9% / 2.0%; Not Defined nan% / 0.0%; Voucher 4.3% / 3.5%.
  - 2018 covers Jan-Aug only, so compare payment shares (%) rather than raw counts across years.

## **Q6.3:** How are orders distributed by number of payment installments?

```sql
WITH order_installments AS (
    SELECT p.order_id, MAX(p.payment_installments) AS installments
    FROM payments_view p
    JOIN orders_view o ON p.order_id = o.order_id
    GROUP BY p.order_id
)
SELECT
    installments AS payment_installments,
    COUNT(*) AS total_orders,
    ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 2) AS pct_of_orders
FROM order_installments
GROUP BY installments
ORDER BY installments ASC;
```

---

```sql
-- MEDIAN(p.payment_installments)
SELECT
    p.payment_type,
    COUNT(DISTINCT p.order_id) AS total_orders,
    ROUND(AVG(p.payment_installments), 2) AS avg_installments,
    ROUND(100.0 * AVG(CASE WHEN p.payment_installments > 1 THEN 1 ELSE 0 END), 2) AS pct_multi_installment
FROM payments_view p
JOIN orders_view o ON p.order_id = o.order_id
GROUP BY p.payment_type
ORDER BY total_orders DESC;
```

---

Insights:
  - Single-payment orders are 48.6% of orders; 51.4% use installments.
  - 185 orders use more than 12 installments and 2 orders have 0 installments (invalid); both are kept in the table rather than dropped.
  - Credit card: mean 3.5 installments (median 3.0); 66.81% of credit-card payments are multi-installment.

## Recommendations

Each recommendation below is printed from the metrics computed above, so numbers always match the data. Where the dataset cannot prove an effect, the item is framed as a test.

1. Fix delivery reliability where it fails: Alagoas (78.54%), Maranhão (82.47%), Sergipe (84.64%), Piauí (86.11%), Ceará (86.17%) have the lowest on-time rates (national 93.2%). Review carrier SLAs and routing for these states first.

2. Do not change the estimate algorithm without a trigger: this dataset has no conversion data, so it cannot show whether long estimates cost sales. Change it only if a competitor promises faster and delivers on it. Meanwhile, reduce the 6.79% late rate: late orders score 2.02 points lower on average.

3. Freight is lower and delivery faster when the seller is in the buyer's state (BRL 15.36 vs 26.98; 7.92 vs 15.12 days). Recruit sellers or stage inventory in high-freight regions (Paraíba, Rondônia, Piauí) rather than assuming volume alone lowers cost.

4. All top-5 states by order value sit in the lower half of freight cost; focus freight work on remote, high-freight states instead.

5. Credit card is used in 76.9% of orders and 48.6% of orders are single-payment. Run an A/B test of cashback or vouchers on a small installment-heavy segment, and check cost against margin data first (not in this dataset).

6. Schedule promotions and support staffing for the Afternoon window and Mondays, the busiest slots in the data.

7. Only 3.03% of customers ordered more than once. Treat repeat purchase as the main growth lever to study (needs cohort and marketing data).

8. Capture cost of goods, shipping cost paid by Target, sessions/cart data and returns. Without them margin, conversion and true LTV cannot be answered.

## **Q7:** Key Metrics Summary (Jan-Aug 2017 vs Jan-Aug 2018)

```sql
-- MEDIAN(d.time_to_deliver_days)
WITH jan_aug AS (
    SELECT * FROM orders_view
    WHERE EXTRACT(MONTH FROM order_purchase_timestamp) BETWEEN 1 AND 8
),
review_scores AS (
    SELECT order_id, AVG(review_score) AS review_score FROM reviews_view GROUP BY order_id
),
item_totals AS (
    SELECT order_id, SUM(price) AS price, SUM(freight_value) AS freight FROM order_items_view GROUP BY order_id
)
SELECT
    EXTRACT(YEAR FROM o.order_purchase_timestamp) AS year,
    COUNT(*) AS orders_placed,
    ROUND(100.0 * AVG(CASE WHEN o.order_status IN ('canceled', 'unavailable') THEN 1 ELSE 0 END), 2) AS canceled_or_unavailable_pct,
    ROUND(SUM(CASE WHEN o.order_status NOT IN ('canceled', 'unavailable') THEN v.price ELSE 0 END), 2) AS gmv_brl,
    ROUND(AVG(CASE WHEN o.order_status NOT IN ('canceled', 'unavailable') THEN v.price END), 2) AS avg_order_value_brl,
    ROUND(AVG(CASE WHEN o.order_status NOT IN ('canceled', 'unavailable') THEN v.freight END), 2) AS avg_freight_per_order_brl,
    ROUND(100.0 * SUM(CASE WHEN o.order_status NOT IN ('canceled', 'unavailable') THEN v.freight ELSE 0 END) 
            / NULLIF(SUM(CASE WHEN o.order_status NOT IN ('canceled', 'unavailable') THEN v.price ELSE 0 END), 0), 2) AS freight_pct_of_price,
    ROUND(AVG(d.time_to_deliver_days), 2) AS avg_delivery_days,
    ROUND(100.0 * AVG(CASE WHEN d.order_id IS NOT NULL AND d.diff_estimated_delivery >= 0 THEN 1 WHEN d.order_id IS NOT NULL THEN 0 END), 2) AS on_time_or_early_pct,
    ROUND(AVG(r.review_score), 2) AS avg_review_score
FROM jan_aug o
LEFT JOIN item_totals v ON o.order_id = v.order_id
LEFT JOIN delivery_metrics_view d ON o.order_id = d.order_id
LEFT JOIN review_scores r ON o.order_id = r.order_id
GROUP BY year
ORDER BY year ASC;
```

---

```sql
WITH customer_orders AS (
    SELECT
        c.customer_unique_id,
        COUNT(DISTINCT o.order_id) AS orders,
        SUM(ov.order_price) AS spend
    FROM orders_view o
    JOIN customers_view c ON o.customer_id = c.customer_id
    JOIN order_values_view ov ON o.order_id = ov.order_id
    GROUP BY c.customer_unique_id
)
SELECT
    COUNT(*) AS unique_customers,
    ROUND(100.0 * AVG(CASE WHEN orders > 1 THEN 1 ELSE 0 END), 2) AS repeat_customer_pct,
    ROUND(AVG(orders), 3) AS orders_per_customer,
    ROUND(AVG(spend), 2) AS revenue_per_customer_brl
FROM customer_orders;
```

---

Insights:
  - 3.03% of 94,703 unique customers ordered more than once in the window (by `customer_unique_id`).
  - Revenue per customer of BRL 142.02 is a window-only proxy, not lifetime value.
  - Percent-change rows for rates (cancel %, on-time %, freight %) are relative changes; read the absolute values alongside them.

### Data notes and limitations

* Orders from 2016 and Sep-Oct 2018 are excluded (partial coverage). Only two calendar years remain, so no yearly trend is claimed.
* `freight_value` is the shipping charge on an item and not Target's logistics cost. No cost, margin or seller-payout data exists, so profitability is not assessed.
* State rankings require at least `MIN_ORDERS_PER_STATE` orders.
* Delivery metrics cover delivered orders only; delivery gap uses calendar dates, delivery time uses 24-hour periods.
* Payment category `UPI` replaces `boleto` in this file.
* State-level freight and delivery differences are not controlled for product weight or category mix.